# RQ3 cross-assessment prediction

Original notebook used for item-wise Ridge regression with 30 repeated user-level holdouts.


## Path configuration

The analysis logic below is unchanged from the research notebook. Theta CSVs
are generated model outputs, not raw input data: train the selected model and
run its CP2 and CP4 diagnosis commands first. Then replace every
`REPLACE_WITH_...` value with the named generated output or authorized target
file. Paths may be absolute or repository-root-relative. Analysis outputs are
written under the repository's `outputs/` directory.


# Experiment 3 — S1: 30 Repeated User-Level Holdouts

This notebook evaluates whether mastery vectors from the corresponding
learner states contain information useful for estimating **Mid** and **Final**
item scores under S1.

- Mid: S1 Group-2 theta → Mid scores
- Final: S1 Group-4 theta → Final scores
- Regression head: item-wise Ridge regression
- Evaluation: 30 repeated 70/30 user-level random splits

The evaluation implementation is identical to the S2 and S3 notebooks.
Only the input paths and setting label differ.


In [ ]:

from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from IPython.display import display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)


In [ ]:

# ============================================================
# SHARED EXPERIMENT SETTINGS
# Keep these values identical for S1, S2, and S3.
# ============================================================

USER_COL = "user_id"
ITEM_COL = "item_id"
SCORE_COL = "score"

N_ROUNDS = 30
TEST_SIZE = 0.30
SEED = 42

RIDGE_ALPHA = 1.0
CLIP_MIN = 0.0
CLIP_MAX = 10.0

MIN_FIT_ROWS = 2
MIN_EVAL_ROWS = 1

RUN_EXPERIMENT = True

print("Evaluation protocol")
print("-" * 60)
print(f"Rounds            : {N_ROUNDS}")
print(f"Fit/Eval ratio    : {1 - TEST_SIZE:.0%}/{TEST_SIZE:.0%}")
print(f"Split unit        : unique {USER_COL}")
print(f"Seed              : {SEED} + round number")
print(f"Regression        : item-wise Ridge")
print(f"Ridge alpha       : {RIDGE_ALPHA}")
print(f"Prediction clip   : [{CLIP_MIN}, {CLIP_MAX}]")


In [ ]:

# ============================================================
# EDIT ONLY THIS CONFIGURATION CELL FOR S1
# ============================================================

# USER CONFIGURATION: replace the four input paths below.

CONFIG = {
    "setting": "S1",
    "model_name": "NCDM",

    # CP2 theta -> full Mid target (filtered by theta user_id during merge)
    "mid_theta": Path("REPLACE_WITH_CP2_THETA_CSV"),
    "mid_target": Path("REPLACE_WITH_MID_SCORE_CSV"),

    # CP4 theta -> full Final target (filtered by theta user_id during merge)
    "final_theta": Path("REPLACE_WITH_CP4_THETA_CSV"),
    "final_target": Path("REPLACE_WITH_FINAL_SCORE_CSV"),

    "output_root": Path("outputs/RQ3_Cross_Assessment/ncdm_poly"),
}

display(pd.DataFrame({
    "key": list(CONFIG.keys()),
    "value": [str(v) for v in CONFIG.values()],
}))


## 1. Load and inspect the input data

Run these cells before the experiment. They print theta, target, merged rows, item counts, score ranges, and descriptive statistics.

In [ ]:

# ============================================================
# DATA LOADING AND VALIDATION
# ============================================================

def detect_theta_columns(df, user_col=USER_COL):
    """
    Detect theta columns using this priority:
    1) columns beginning with 'theta'
    2) K0, K1, ..., K17
    3) remaining numeric columns after excluding metadata
    """
    theta_cols = [
        c for c in df.columns
        if str(c).lower().startswith("theta")
    ]
    if theta_cols:
        return theta_cols

    k_cols = [
        c for c in df.columns
        if re.fullmatch(r"K\d+", str(c))
    ]
    if k_cols:
        return sorted(k_cols, key=lambda c: int(str(c)[1:]))

    metadata_cols = {
        user_col, ITEM_COL, SCORE_COL,
        "group", "exam", "split", "assessment", "target_exam",
        "item_name", "student_id", "real_student_id",
    }
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    theta_cols = [c for c in numeric_cols if c not in metadata_cols]

    if not theta_cols:
        raise ValueError(
            "No theta columns found. Expected theta_* columns, "
            "K0-K17 columns, or numeric mastery columns."
        )
    return theta_cols


def load_theta(path, user_col=USER_COL):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Theta file not found: {path}")

    df = pd.read_csv(path)

    if user_col not in df.columns:
        raise ValueError(f"Theta file must contain '{user_col}': {path}")

    duplicated = df.loc[df[user_col].duplicated(), user_col].unique()
    if len(duplicated):
        raise ValueError(
            f"Theta file contains duplicated {user_col}. "
            f"Examples: {duplicated[:10].tolist()}"
        )

    theta_cols = detect_theta_columns(df, user_col=user_col)

    for col in theta_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    missing_count = int(df[theta_cols].isna().sum().sum())
    if missing_count:
        raise ValueError(
            f"Theta columns contain {missing_count} missing or non-numeric values: {path}"
        )

    return df, theta_cols


def load_target(
    path,
    user_col=USER_COL,
    item_col=ITEM_COL,
    score_col=SCORE_COL,
):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Target file not found: {path}")

    df = pd.read_csv(path)

    required = [user_col, item_col, score_col]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Target file is missing columns {missing}: {path}")

    df[score_col] = pd.to_numeric(df[score_col], errors="coerce")

    missing_scores = int(df[score_col].isna().sum())
    if missing_scores:
        raise ValueError(
            f"Target contains {missing_scores} missing or non-numeric scores: {path}"
        )

    duplicated_pairs = df.duplicated([user_col, item_col], keep=False)
    if duplicated_pairs.any():
        examples = (
            df.loc[duplicated_pairs, [user_col, item_col]]
            .drop_duplicates()
            .head(10)
        )
        raise ValueError(
            "Target contains duplicated (user_id, item_id) rows.\n"
            f"Examples:\n{examples}"
        )

    return df


def merge_theta_target(
    theta_df,
    target_df,
    theta_cols,
    user_col=USER_COL,
):
    theta_users = set(theta_df[user_col])
    target_users = set(target_df[user_col])

    # Full target files may contain students outside the current theta set.
    # Require every theta user to have target responses, then retain only
    # target rows belonging to theta users.
    missing_target_users = sorted(theta_users - target_users)
    if missing_target_users:
        raise ValueError(
            f"{len(missing_target_users)} theta users are missing from the target file. "
            f"Examples: {missing_target_users[:20]}"
        )

    matched_target_df = target_df[
        target_df[user_col].isin(theta_users)
    ].copy()

    merged = matched_target_df.merge(
        theta_df[[user_col] + theta_cols],
        on=user_col,
        how="left",
        validate="many_to_one",
    )

    if merged[theta_cols].isna().any().any():
        raise ValueError("Merged data contains missing theta values.")

    return merged


def inspect_exam_inputs(
    theta_path,
    target_path,
    exam_name,
):
    """
    Load, validate, print, and return theta, target, and merged data.
    """
    theta_df, theta_cols = load_theta(theta_path)
    target_df = load_target(target_path)
    merged_df = merge_theta_target(theta_df, target_df, theta_cols)

    theta_users = set(theta_df[USER_COL])
    target_users = set(target_df[USER_COL])

    print("\n" + "=" * 100)
    print(f"{exam_name.upper()} INPUT AUDIT")
    print("=" * 100)
    print(f"Theta path       : {theta_path}")
    print(f"Target path      : {target_path}")
    print(f"Theta rows       : {len(theta_df)}")
    print(f"Theta users      : {theta_df[USER_COL].nunique()}")
    print(f"Theta dimensions : {len(theta_cols)}")
    print(f"Target rows      : {len(target_df)}")
    print(f"Target users     : {target_df[USER_COL].nunique()}")
    print(f"Target items     : {target_df[ITEM_COL].nunique()}")
    print(f"Merged rows      : {len(merged_df)}")
    print(f"Merged users     : {merged_df[USER_COL].nunique()}")
    print(f"Score range      : {target_df[SCORE_COL].min():.3f} to {target_df[SCORE_COL].max():.3f}")
    print(f"Target-only users: {len(target_users - theta_users)}")
    print(f"Theta-only users : {len(theta_users - target_users)}")
    print(f"Theta columns    : {theta_cols}")

    print("\nTheta preview")
    display(theta_df.head(10))

    print("\nTheta descriptive statistics")
    display(theta_df[theta_cols].describe().T)

    print("\nTarget preview")
    display(target_df.head(20))

    print("\nTarget counts by item")
    display(
        target_df.groupby(ITEM_COL)
        .agg(
            rows=(SCORE_COL, "size"),
            users=(USER_COL, "nunique"),
            mean_score=(SCORE_COL, "mean"),
            std_score=(SCORE_COL, "std"),
            min_score=(SCORE_COL, "min"),
            max_score=(SCORE_COL, "max"),
        )
        .reset_index()
    )

    print("\nMerged data preview used by Ridge")
    display(merged_df.head(20))

    return {
        "theta": theta_df,
        "theta_cols": theta_cols,
        "target": target_df,
        "merged": merged_df,
    }


In [ ]:

# ============================================================
# LOAD AND INSPECT ALL INPUT DATA BEFORE RUNNING THE EXPERIMENT
# ============================================================

mid_data = inspect_exam_inputs(
    theta_path=CONFIG["mid_theta"],
    target_path=CONFIG["mid_target"],
    exam_name="Mid",
)

final_data = inspect_exam_inputs(
    theta_path=CONFIG["final_theta"],
    target_path=CONFIG["final_target"],
    exam_name="Final",
)

if mid_data["theta_cols"] != final_data["theta_cols"]:
    raise ValueError(
        "Mid and Final theta columns are not identical.\n"
        f"Mid  : {mid_data['theta_cols']}\n"
        f"Final: {final_data['theta_cols']}"
    )

audit_summary = pd.DataFrame([
    {
        "exam": "Mid",
        "theta_rows": len(mid_data["theta"]),
        "theta_users": mid_data["theta"][USER_COL].nunique(),
        "target_rows": len(mid_data["target"]),
        "target_users": mid_data["target"][USER_COL].nunique(),
        "target_items": mid_data["target"][ITEM_COL].nunique(),
        "merged_rows": len(mid_data["merged"]),
        "merged_users": mid_data["merged"][USER_COL].nunique(),
    },
    {
        "exam": "Final",
        "theta_rows": len(final_data["theta"]),
        "theta_users": final_data["theta"][USER_COL].nunique(),
        "target_rows": len(final_data["target"]),
        "target_users": final_data["target"][USER_COL].nunique(),
        "target_items": final_data["target"][ITEM_COL].nunique(),
        "merged_rows": len(final_data["merged"]),
        "merged_users": final_data["merged"][USER_COL].nunique(),
    },
])

print("\nCombined audit summary")
display(audit_summary)


## 2. Define the repeated-holdout evaluation

In [ ]:

# ============================================================
# REPEATED USER-LEVEL HOLDOUT WITH ITEM-WISE RIDGE
# ============================================================

def run_repeated_itemwise_ridge(
    theta_df,
    target_df,
    theta_cols,
    setting,
    model_name,
    target_exam,
    n_rounds=N_ROUNDS,
    test_size=TEST_SIZE,
    seed=SEED,
    ridge_alpha=RIDGE_ALPHA,
    clip_min=CLIP_MIN,
    clip_max=CLIP_MAX,
    min_fit_rows=MIN_FIT_ROWS,
    min_eval_rows=MIN_EVAL_ROWS,
):
    """
    For each round:
      1) split unique users into 70% fit and 30% evaluation groups;
      2) fit one Ridge model per item;
      3) predict held-out users;
      4) pool all valid prediction errors and compute one MAE per round.

    Mean and sample SD are computed from the 30 round-level MAEs.
    """
    merged = merge_theta_target(theta_df, target_df, theta_cols)
    users = np.array(sorted(merged[USER_COL].unique()))

    if len(users) < 3:
        raise ValueError(f"At least 3 unique users are required; found {len(users)}.")

    print("\n" + "=" * 100)
    print(f"RUNNING {setting} | {model_name} | {target_exam}")
    print("=" * 100)
    print(f"Rows          : {len(merged)}")
    print(f"Unique users  : {len(users)}")
    print(f"Unique items  : {merged[ITEM_COL].nunique()}")
    print(f"Rounds        : {n_rounds}")
    print(f"Test size     : {test_size}")
    print(f"Ridge alpha   : {ridge_alpha}")

    per_round_records = []
    per_item_records = []
    prediction_records = []
    skipped_records = []

    for round_id in range(1, n_rounds + 1):
        fit_user_array, eval_user_array = train_test_split(
            users,
            test_size=test_size,
            random_state=seed + round_id,
            shuffle=True,
        )

        fit_users = set(fit_user_array)
        eval_users = set(eval_user_array)

        overlap = fit_users & eval_users
        if overlap:
            raise RuntimeError(
                f"Round {round_id}: fit/evaluation user overlap: {sorted(overlap)}"
            )

        fit_df = merged[merged[USER_COL].isin(fit_users)].copy()
        eval_df = merged[merged[USER_COL].isin(eval_users)].copy()

        round_abs_errors = []

        for item_id in sorted(merged[ITEM_COL].unique()):
            item_fit = fit_df[fit_df[ITEM_COL] == item_id].copy()
            item_eval = eval_df[eval_df[ITEM_COL] == item_id].copy()

            n_fit = len(item_fit)
            n_eval = len(item_eval)

            if n_fit < min_fit_rows or n_eval < min_eval_rows:
                skipped_records.append({
                    "setting": setting,
                    "model_name": model_name,
                    "target_exam": target_exam,
                    "round": round_id,
                    "item_id": item_id,
                    "n_fit_rows": n_fit,
                    "n_eval_rows": n_eval,
                    "reason": "insufficient rows",
                })
                continue

            X_fit = item_fit[theta_cols].to_numpy(dtype=float)
            y_fit = item_fit[SCORE_COL].to_numpy(dtype=float)

            X_eval = item_eval[theta_cols].to_numpy(dtype=float)
            y_eval = item_eval[SCORE_COL].to_numpy(dtype=float)

            model = Ridge(alpha=ridge_alpha)
            model.fit(X_fit, y_fit)

            y_pred = model.predict(X_eval)
            y_pred = np.clip(y_pred, clip_min, clip_max)
            abs_errors = np.abs(y_eval - y_pred)

            per_item_records.append({
                "setting": setting,
                "model_name": model_name,
                "target_exam": target_exam,
                "round": round_id,
                "item_id": item_id,
                "mae": float(abs_errors.mean()),
                "n_fit_rows": n_fit,
                "n_eval_rows": n_eval,
            })

            for row_index, true_value, predicted_value, error in zip(
                item_eval.index, y_eval, y_pred, abs_errors
            ):
                prediction_records.append({
                    "setting": setting,
                    "model_name": model_name,
                    "target_exam": target_exam,
                    "round": round_id,
                    USER_COL: item_eval.loc[row_index, USER_COL],
                    ITEM_COL: item_id,
                    "y_true": float(true_value),
                    "y_pred": float(predicted_value),
                    "abs_error": float(error),
                })

            round_abs_errors.extend(abs_errors.tolist())

        round_mae = (
            float(np.mean(round_abs_errors))
            if round_abs_errors
            else np.nan
        )

        per_round_records.append({
            "setting": setting,
            "model_name": model_name,
            "target_exam": target_exam,
            "round": round_id,
            "mae": round_mae,
            "n_predictions": len(round_abs_errors),
            "n_fit_users": len(fit_users),
            "n_eval_users": len(eval_users),
        })

    per_round = pd.DataFrame(per_round_records)
    per_item = pd.DataFrame(per_item_records)
    predictions = pd.DataFrame(prediction_records)
    skipped = pd.DataFrame(skipped_records)

    valid_round_mae = per_round["mae"].dropna()

    summary = pd.DataFrame([{
        "setting": setting,
        "model_name": model_name,
        "target_exam": target_exam,
        "mean_mae": float(valid_round_mae.mean()) if len(valid_round_mae) else np.nan,
        "std_mae": float(valid_round_mae.std(ddof=1)) if len(valid_round_mae) > 1 else np.nan,
        "n_valid_rounds": int(valid_round_mae.size),
        "n_requested_rounds": int(n_rounds),
        "n_predictions_mean": float(per_round["n_predictions"].mean()),
        "n_predictions_min": int(per_round["n_predictions"].min()),
        "n_predictions_max": int(per_round["n_predictions"].max()),
        "n_users": int(len(users)),
        "n_items": int(merged[ITEM_COL].nunique()),
        "n_skipped_item_rounds": int(len(skipped)),
        "ridge_alpha": float(ridge_alpha),
    }])

    print("\nExam summary")
    display(summary)

    if len(skipped):
        print(f"Skipped item-round combinations: {len(skipped)}")
        display(skipped.head(20))
    else:
        print("No item-round combinations were skipped.")

    return {
        "summary": summary,
        "per_round": per_round,
        "per_item": per_item,
        "predictions": predictions,
        "skipped": skipped,
        "merged": merged,
    }


def compute_pooled_mid_final_average(
    mid_predictions,
    final_predictions,
    setting,
    model_name,
):
    """
    Preserve the original protocol:
    for each round, pool all Mid and Final absolute errors first,
    then compute one Avg MAE. Mean and sample SD are calculated
    across the 30 pooled round-level MAEs.
    """
    pooled = pd.concat(
        [mid_predictions, final_predictions],
        ignore_index=True,
    )

    if pooled.empty:
        raise ValueError("No Mid or Final predictions are available for Avg calculation.")

    avg_per_round = (
        pooled
        .groupby(["setting", "model_name", "round"], as_index=False)
        .agg(
            mae=("abs_error", "mean"),
            n_predictions=("abs_error", "size"),
        )
    )

    avg_per_round["target_exam"] = "Avg"
    avg_per_round["n_fit_users"] = np.nan
    avg_per_round["n_eval_users"] = np.nan

    avg_per_round = avg_per_round[
        [
            "setting", "model_name", "target_exam", "round",
            "mae", "n_predictions", "n_fit_users", "n_eval_users",
        ]
    ]

    valid_round_mae = avg_per_round["mae"].dropna()

    avg_summary = pd.DataFrame([{
        "setting": setting,
        "model_name": model_name,
        "target_exam": "Avg",
        "mean_mae": float(valid_round_mae.mean()),
        "std_mae": float(valid_round_mae.std(ddof=1)),
        "n_valid_rounds": int(valid_round_mae.size),
        "n_requested_rounds": int(N_ROUNDS),
        "n_predictions_mean": float(avg_per_round["n_predictions"].mean()),
        "n_predictions_min": int(avg_per_round["n_predictions"].min()),
        "n_predictions_max": int(avg_per_round["n_predictions"].max()),
        "n_users": np.nan,
        "n_items": np.nan,
        "n_skipped_item_rounds": np.nan,
        "ridge_alpha": float(RIDGE_ALPHA),
    }])

    return avg_summary, avg_per_round


## 3. Run the experiment

In [ ]:

# ============================================================
# RUN THE COMPLETE EXPERIMENT
# ============================================================

def run_experiment(config):
    setting = config["setting"]
    model_name = config["model_name"]

    print("\n" + "#" * 100)
    print(f"EXPERIMENT 3 | {setting} | {model_name}")
    print("#" * 100)

    mid_result = run_repeated_itemwise_ridge(
        theta_df=mid_data["theta"],
        target_df=mid_data["target"],
        theta_cols=mid_data["theta_cols"],
        setting=setting,
        model_name=model_name,
        target_exam="Mid",
    )

    final_result = run_repeated_itemwise_ridge(
        theta_df=final_data["theta"],
        target_df=final_data["target"],
        theta_cols=final_data["theta_cols"],
        setting=setting,
        model_name=model_name,
        target_exam="Final",
    )

    avg_summary, avg_per_round = compute_pooled_mid_final_average(
        mid_predictions=mid_result["predictions"],
        final_predictions=final_result["predictions"],
        setting=setting,
        model_name=model_name,
    )

    summary = pd.concat(
        [
            mid_result["summary"],
            final_result["summary"],
            avg_summary,
        ],
        ignore_index=True,
    )

    per_round = pd.concat(
        [
            mid_result["per_round"],
            final_result["per_round"],
            avg_per_round,
        ],
        ignore_index=True,
    )

    per_item = pd.concat(
        [
            mid_result["per_item"],
            final_result["per_item"],
        ],
        ignore_index=True,
    )

    predictions = pd.concat(
        [
            mid_result["predictions"],
            final_result["predictions"],
        ],
        ignore_index=True,
    )

    skipped = pd.concat(
        [
            mid_result["skipped"],
            final_result["skipped"],
        ],
        ignore_index=True,
    )

    output_dir = (
        Path(config["output_root"])
        / setting
        / model_name
    )
    output_dir.mkdir(parents=True, exist_ok=True)

    summary.to_csv(output_dir / "summary.csv", index=False)
    per_round.to_csv(output_dir / "per_round.csv", index=False)
    per_item.to_csv(output_dir / "per_item.csv", index=False)
    predictions.to_csv(output_dir / "predictions.csv", index=False)
    skipped.to_csv(output_dir / "skipped.csv", index=False)

    audit_dir = output_dir / "audit"
    audit_dir.mkdir(parents=True, exist_ok=True)

    mid_data["theta"].to_csv(audit_dir / "mid_theta.csv", index=False)
    mid_data["target"].to_csv(audit_dir / "mid_target.csv", index=False)
    mid_data["merged"].to_csv(audit_dir / "mid_merged_used_by_ridge.csv", index=False)

    final_data["theta"].to_csv(audit_dir / "final_theta.csv", index=False)
    final_data["target"].to_csv(audit_dir / "final_target.csv", index=False)
    final_data["merged"].to_csv(audit_dir / "final_merged_used_by_ridge.csv", index=False)

    print("\nSaved outputs to:")
    print(output_dir)

    return {
        "summary": summary,
        "per_round": per_round,
        "per_item": per_item,
        "predictions": predictions,
        "skipped": skipped,
        "output_dir": output_dir,
    }


if RUN_EXPERIMENT:
    results = run_experiment(CONFIG)
else:
    print("RUN_EXPERIMENT is False. Check CONFIG and set it to True.")


## 4. Inspect the results

In [ ]:

# ============================================================
# PAPER-STYLE SUMMARY
# ============================================================

if "results" in globals():
    summary_display = results["summary"].copy()
    summary_display["MAE"] = summary_display.apply(
        lambda row: f"{row['mean_mae']:.3f} ± {row['std_mae']:.3f}",
        axis=1,
    )
    display(
        summary_display[
            [
                "setting",
                "model_name",
                "target_exam",
                "MAE",
                "n_valid_rounds",
                "n_predictions_mean",
                "n_skipped_item_rounds",
            ]
        ]
    )


In [ ]:

# ============================================================
# CHECK THE 30 ROUND-LEVEL MAEs
# ============================================================

if "results" in globals():
    round_pivot = (
        results["per_round"]
        .pivot(index="round", columns="target_exam", values="mae")
        .reset_index()
    )
    display(round_pivot)

    print("\nRound-level descriptive statistics")
    display(
        results["per_round"]
        .groupby("target_exam")["mae"]
        .describe()
    )


In [ ]:

# ============================================================
# PLOT ROUND-LEVEL MAE
# ============================================================

if "results" in globals():
    plt.figure(figsize=(11, 5))

    for exam_name, exam_df in results["per_round"].groupby("target_exam"):
        plt.plot(
            exam_df["round"],
            exam_df["mae"],
            marker="o",
            linewidth=1,
            label=exam_name,
        )

    plt.xlabel("Round")
    plt.ylabel("MAE")
    plt.title(
        f"{CONFIG['setting']} | {CONFIG['model_name']} | "
        "30 repeated user-level holdouts"
    )
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:

# ============================================================
# ITEM-LEVEL RESULTS
# ============================================================

if "results" in globals():
    item_summary = (
        results["per_item"]
        .groupby(["target_exam", "item_id"], as_index=False)
        .agg(
            mean_mae=("mae", "mean"),
            std_mae=("mae", "std"),
            valid_rounds=("round", "nunique"),
            mean_fit_rows=("n_fit_rows", "mean"),
            mean_eval_rows=("n_eval_rows", "mean"),
        )
        .sort_values(["target_exam", "mean_mae"], ascending=[True, False])
    )
    display(item_summary)


In [ ]:

# ============================================================
# LARGEST PREDICTION ERRORS
# ============================================================

if "results" in globals():
    display(
        results["predictions"]
        .sort_values("abs_error", ascending=False)
        .head(30)
    )


In [ ]:

# ============================================================
# SKIPPED ITEM-ROUND CHECK
# ============================================================

if "results" in globals():
    if results["skipped"].empty:
        print("No item-round combinations were skipped.")
    else:
        print(f"Skipped rows: {len(results['skipped'])}")
        display(results["skipped"])
